In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [28]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent


In [4]:
#loader
loader=PyPDFLoader("../AI_Vehicle_Management_System_Interview_Guide.pdf")
docs=loader.load()
len(docs)

5

In [5]:
#splitter
splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_data=splitter.split_documents(docs)
len(splitted_data)

14

In [6]:
#embedding
embeddings=GoogleGenerativeAIEmbeddings(model="gemini-embedding-2")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [7]:
#Store Vector Embedding in Vector db
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_data,
    embedding=embeddings
  
)

In [12]:
same_record=vector_store.similarity_search("Project Explanation")
same_record[0].page_content

'AI-Enabled Vehicle Management System\nComplete Project Explanation & Technical Interview Preparation Guide (All 24 Questions)\nTech Stack: Java • Spring Boot • Oracle DB • Spring Data JPA • Groq API • JUnit • Postman\nPart 1: Project Overview, Problem & Impact\n(1) Which problem does this project solve?\nTraditional vehicle management systems only store static inventory records and force users to\nmanually filter through dozens of options. This project solves that by combining reliable inventory\nmanagement with an AI recommendation engine that matches users to the right vehicle\nbased on natural preferences.\n(2) What is your problem statement?\nTo build a scalable, fault-tolerant RESTful backend that manages vehicle inventory efficiently\nwhile providing fast, intelligent vehicle recommendations without manual search fatigue.\n(3) What features did you add to this project?\nFull Vehicle CRUD: Create, read, update, and delete vehicle records via REST endpoints.'

In [23]:
#agent=tools,llm,prompt
@tool
def retriever_tool(query:str):
    """
     The tool can help you to retrieve 
     the relevant data of the pdf documents and these documents have details about project reports

    """
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    
    for doc in docs:
        context+=doc.page_content+"\n"
        print(context)
        



In [25]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [35]:
System_prompt="""
         You are a helpful assistant and provide answered based on the context for user questions using retreived context.
         Always use the `retriever_tool` tool for questions requiring external knowledge

"""

In [38]:
agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=System_prompt
)

In [41]:
query="explain only technical skill"
response=agent.invoke({"messages":[{"role":"user","content":"query"}]})
result=response["messages"][-1].content

AI-Enabled Vehicle Management System
Complete Project Explanation & Technical Interview Preparation Guide (All 24 Questions)
Tech Stack: Java • Spring Boot • Oracle DB • Spring Data JPA • Groq API • JUnit • Postman
Part 1: Project Overview, Problem & Impact
(1) Which problem does this project solve?
Traditional vehicle management systems only store static inventory records and force users to
manually filter through dozens of options. This project solves that by combining reliable inventory
management with an AI recommendation engine that matches users to the right vehicle
based on natural preferences.
(2) What is your problem statement?
To build a scalable, fault-tolerant RESTful backend that manages vehicle inventory efficiently
while providing fast, intelligent vehicle recommendations without manual search fatigue.
(3) What features did you add to this project?
Full Vehicle CRUD: Create, read, update, and delete vehicle records via REST endpoints.

AI-Enabled Vehicle Management Syste